## 转为灰度图

In [10]:
from PIL import Image
import os

# Source and destination directories
source_dir = '/root/exp/datasets/xijing_split/trainB_crop'
dest_dir = '/root/exp/datasets/xijing_split/trainB_crop_gray'

# Create destination directory if it doesn't exist
os.makedirs(dest_dir, exist_ok=True)

# Process all images in the source directory
for filename in os.listdir(source_dir):
    if filename.lower().endswith(('.png', '.jpg', '.jpeg', '.bmp', '.tiff')):
        # Load image
        img_path = os.path.join(source_dir, filename)
        img = Image.open(img_path)
        
        # Convert to grayscale
        gray_img = img.convert('L')
        
        # Save to destination directory with same filename
        dest_path = os.path.join(dest_dir, filename)
        gray_img.save(dest_path)
        
        # print(f"Converted {filename} to grayscale")

print("All images converted to grayscale successfully!")

All images converted to grayscale successfully!


## 随机划分数据集

In [3]:
import random
import shutil
import os

# Set random seed for reproducibility
random.seed(42)

# Source and destination directories
# source_dir = '/root/exp/datasets/xijing_split/test/test_unpaired_LR_feiyinuo_all'
# dest_dir = '/root/exp/datasets/xijing_split/test/test_unpaired_LR_feiyinuo_part0.1'

source_dir = '/root/exp/datasets/xijing_split/test/test_unpaired_LR_feiyinuo_all'
dest_dir = '/root/exp/datasets/xijing_split/test/test_unpaired_LR_feiyinuo_part0.1'

# Create destination directory if it doesn't exist
os.makedirs(dest_dir, exist_ok=True)

# Get all image files from source directory
image_files = [f for f in os.listdir(source_dir) if f.lower().endswith(('.png', '.jpg', '.jpeg', '.bmp', '.tiff'))]

# Calculate 1/10 of total images
num_to_select = len(image_files) // 10

# Randomly select 1/10 of the images
selected_files = random.sample(image_files, num_to_select)

# Copy selected files to destination directory
for filename in selected_files:
    src_path = os.path.join(source_dir, filename)
    dest_path = os.path.join(dest_dir, filename)
    shutil.copy2(src_path, dest_path)

print(f"Selected {len(selected_files)} images out of {len(image_files)} total images")
print(f"Images copied to {dest_dir}")

Selected 720 images out of 7202 total images
Images copied to /root/exp/datasets/xijing_split/test/test_unpaired_LR_feiyinuo_part0.1


## crop test

### HR-LR 配对数据集处理

In [2]:
#!/usr/bin/env python3
"""
图像裁剪程序
使用 resize_and_crop 方式处理图像：先 resize 到 286x286，然后 crop 到 256x256
确保 HR 和 LR 图像使用相同的裁剪参数
"""

import os
import sys
import argparse
from PIL import Image
import random
import numpy as np

# 添加当前目录到 Python 路径，以便导入 data.base_dataset
# sys.path.append(os.path.dirname(os.path.abspath(__file__)))

from data.base_dataset import get_params, __crop
import torchvision.transforms as transforms


def resize_and_crop_image(img_path, load_size, crop_size, crop_pos=None):
    """
    对图像进行 resize_and_crop 操作
    
    Args:
        img_path: 图像路径
        load_size: resize 后的尺寸
        crop_size: crop 后的尺寸  
        crop_pos: 裁剪位置 (x, y)，如果为 None 则随机裁剪
        
    Returns:
        processed_img: 处理后的 PIL 图像
        crop_pos: 实际使用的裁剪位置
    """
    # 打开图像
    img = Image.open(img_path).convert('RGB')
    
    # Resize 到 load_size x load_size
    method = Image.LANCZOS
    img_resized = img.resize((load_size, load_size), method)
    
    # 确定裁剪位置
    if crop_pos is None:
        # 随机选择裁剪位置
        max_offset = load_size - crop_size
        x = random.randint(0, max_offset) if max_offset > 0 else 0
        y = random.randint(0, max_offset) if max_offset > 0 else 0
        crop_pos = (x, y)
    
    # 执行裁剪
    x, y = crop_pos
    img_cropped = img_resized.crop((x, y, x + crop_size, y + crop_size))
    
    return img_cropped, crop_pos


def process_paired_images(hr_dir, lr_dir, hr_output_dir, lr_output_dir, load_size=286, crop_size=256):
    """
    处理成对的 HR 和 LR 图像，确保使用相同的裁剪参数
    
    Args:
        hr_dir: HR 图像输入目录
        lr_dir: LR 图像输入目录  
        hr_output_dir: HR 图像输出目录
        lr_output_dir: LR 图像输出目录
        load_size: resize 尺寸
        crop_size: crop 尺寸
    """
    # 创建输出目录
    os.makedirs(hr_output_dir, exist_ok=True)
    os.makedirs(lr_output_dir, exist_ok=True)
    
    # 获取 HR 图像列表
    hr_files = [f for f in os.listdir(hr_dir) if f.lower().endswith(('.png', '.jpg', '.jpeg'))]
    hr_files.sort()
    
    processed_count = 0
    
    for hr_file in hr_files:
        # 构造对应的 LR 文件名
        lr_file = hr_file.replace('_HR_', '_LR_')
        
        hr_path = os.path.join(hr_dir, hr_file)
        lr_path = os.path.join(lr_dir, lr_file)
        
        # 检查对应的 LR 文件是否存在
        if not os.path.exists(lr_path):
            print(f"警告: 找不到对应的 LR 文件: {lr_path}")
            continue
            
        try:
            # 处理 HR 图像（随机裁剪）
            hr_img, crop_pos = resize_and_crop_image(hr_path, load_size, crop_size)
            
            # 处理 LR 图像（使用相同的裁剪位置）  
            lr_img, _ = resize_and_crop_image(lr_path, load_size, crop_size, crop_pos)
            
            # 保存处理后的图像
            hr_output_path = os.path.join(hr_output_dir, hr_file)
            lr_output_path = os.path.join(lr_output_dir, lr_file)
            
            hr_img.save(hr_output_path)
            lr_img.save(lr_output_path)
            
            processed_count += 1
            print(f"已处理: {hr_file} -> crop_pos: {crop_pos}")
            
        except Exception as e:
            print(f"处理 {hr_file} 时出错: {str(e)}")
            continue
    
    print(f"\n处理完成！共处理了 {processed_count} 对图像")

In [3]:
# 调用示例 - HR-LR 配对数据集处理

# 设置参数
hr_dir = '/root/exp/us-hand-to-large/datasets/xijing_split/test/test_semi_paired_HR'
lr_dir = '/root/exp/us-hand-to-large/datasets/xijing_split/test/test_semi_paired_LR'
hr_output_dir = '/root/exp/us-hand-to-large/datasets/xijing_split/test/test_semi_paired_HR_crop111'
lr_output_dir = '/root/exp/us-hand-to-large/datasets/xijing_split/test/test_semi_paired_LR_crop111'
load_size = 286
crop_size = 256

print("=== 图像裁剪程序 ===")
print(f"HR 输入目录: {hr_dir}")
print(f"LR 输入目录: {lr_dir}")
print(f"HR 输出目录: {hr_output_dir}")
print(f"LR 输出目录: {lr_output_dir}")
print(f"Load size: {load_size}")
print(f"Crop size: {crop_size}")
print(f"预处理方式: resize_and_crop")
print()

# 检查输入目录是否存在
if not os.path.exists(hr_dir):
    print(f"错误: HR 输入目录不存在: {hr_dir}")
elif not os.path.exists(lr_dir):
    print(f"错误: LR 输入目录不存在: {lr_dir}")
else:
    # 设置随机种子以便复现
    random.seed(42)
    np.random.seed(42)
    
    # 开始处理
    process_paired_images(
        hr_dir, 
        lr_dir,
        hr_output_dir, 
        lr_output_dir,
        load_size,
        crop_size
    )

=== 图像裁剪程序 ===
HR 输入目录: /root/exp/us-hand-to-large/datasets/xijing_split/test/test_semi_paired_HR
LR 输入目录: /root/exp/us-hand-to-large/datasets/xijing_split/test/test_semi_paired_LR
HR 输出目录: /root/exp/us-hand-to-large/datasets/xijing_split/test/test_semi_paired_HR_crop111
LR 输出目录: /root/exp/us-hand-to-large/datasets/xijing_split/test/test_semi_paired_LR_crop111
Load size: 286
Crop size: 256
预处理方式: resize_and_crop

已处理: xijing_HR_104.png -> crop_pos: (20, 3)
已处理: xijing_HR_108.png -> crop_pos: (0, 23)
已处理: xijing_HR_11.png -> crop_pos: (8, 7)
已处理: xijing_HR_116.png -> crop_pos: (7, 4)
已处理: xijing_HR_117.png -> crop_pos: (23, 3)
已处理: xijing_HR_118.png -> crop_pos: (21, 23)
已处理: xijing_HR_120.png -> crop_pos: (28, 17)
已处理: xijing_HR_129.png -> crop_pos: (2, 18)
已处理: xijing_HR_15.png -> crop_pos: (13, 1)
已处理: xijing_HR_150.png -> crop_pos: (0, 2)
已处理: xijing_HR_118.png -> crop_pos: (21, 23)
已处理: xijing_HR_120.png -> crop_pos: (28, 17)
已处理: xijing_HR_129.png -> crop_pos: (2, 18)
已处理: xijing_

### 非配对（单个）数据集处理

In [6]:
def process_single_folder(input_dir, output_dir, load_size=286, crop_size=256):
    """
    处理单个文件夹中的图像，进行 resize_and_crop 操作
    
    Args:
        input_dir: 输入图像目录
        output_dir: 输出图像目录
        load_size: resize 尺寸
        crop_size: crop 尺寸
    """
    # 创建输出目录
    os.makedirs(output_dir, exist_ok=True)
    
    # 获取图像列表
    image_files = [f for f in os.listdir(input_dir) if f.lower().endswith(('.png', '.jpg', '.jpeg'))]
    image_files.sort()
    
    processed_count = 0
    
    for image_file in image_files:
        input_path = os.path.join(input_dir, image_file)
        
        try:
            # 处理图像（随机裁剪）
            processed_img, crop_pos = resize_and_crop_image(input_path, load_size, crop_size)
            
            # 保存处理后的图像（保持原文件名）
            output_path = os.path.join(output_dir, image_file)
            processed_img.save(output_path)
            
            processed_count += 1
            print(f"已处理: {image_file} -> crop_pos: {crop_pos}")
            
        except Exception as e:
            print(f"处理 {image_file} 时出错: {str(e)}")
            continue
    
    print(f"\n处理完成！共处理了 {processed_count} 张图像")

In [ ]:
# 调用示例 - 单个文件夹处理
input_dir = '/root/exp/us-hand-to-large/datasets/xijing_split/trainB'
output_dir = '/root/exp/us-hand-to-large/datasets/xijing_split/trainB_crop'
load_size = 286
crop_size = 256

print("=== 单个文件夹图像裁剪程序 ===")
print(f"输入目录: {input_dir}")
print(f"输出目录: {output_dir}")
print(f"Load size: {load_size}")
print(f"Crop size: {crop_size}")
print(f"预处理方式: resize_and_crop")
print()

# 检查输入目录是否存在
if not os.path.exists(input_dir):
    print(f"错误: 输入目录不存在: {input_dir}")
else:
    # 设置随机种子以便复现
    random.seed(42)
    np.random.seed(42)
    
    # 开始处理
    process_single_folder(
        input_dir,
        output_dir,
        load_size,
        crop_size
    )